In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/MolTrans/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/MolTrans'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/MolTrans/script')



In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils import data
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, roc_curve
import codecs
from subword_nmt.apply_bpe import BPE

# Import the model module
from models import BIN_Interaction_Flat
from config import BIN_config_DBPE

# ==================== Data preprocessing functions ====================
vocab_path_protein = _kp('external/MolTrans/ESPF/protein_codes_uniprot.txt')
bpe_codes_protein = codecs.open(vocab_path_protein)
pbpe = BPE(bpe_codes_protein, merges=-1, separator='')
sub_csv_p = pd.read_csv(_kp('external/MolTrans/ESPF/subword_units_map_uniprot.csv'))
idx2word_p = sub_csv_p['index'].values
words2idx_p = dict(zip(idx2word_p, range(0, len(idx2word_p))))

vocab_path_drug = _kp('external/MolTrans/ESPF/drug_codes_chembl.txt')
bpe_codes_drug = codecs.open(vocab_path_drug)
dbpe = BPE(bpe_codes_drug, merges=-1, separator='')
sub_csv_d = pd.read_csv(_kp('external/MolTrans/ESPF/subword_units_map_chembl.csv'))
idx2word_d = sub_csv_d['index'].values
words2idx_d = dict(zip(idx2word_d, range(0, len(idx2word_d))))

def protein2emb_encoder(x):
    max_p = 545
    t1 = pbpe.process_line(x).split()
    try:
        i1 = np.asarray([words2idx_p[i] for i in t1])
    except:
        i1 = np.array([0])
    
    l = len(i1)
    if l < max_p:
        i = np.pad(i1, (0, max_p - l), 'constant', constant_values=0)
        input_mask = ([1] * l) + ([0] * (max_p - l))
    else:
        i = i1[:max_p]
        input_mask = [1] * max_p
    
    return i, np.asarray(input_mask)

def drug2emb_encoder(x):
    max_d = 50
    t1 = dbpe.process_line(x).split()
    try:
        i1 = np.asarray([words2idx_d[i] for i in t1])
    except:
        i1 = np.array([0])
    
    l = len(i1)
    if l < max_d:
        i = np.pad(i1, (0, max_d - l), 'constant', constant_values=0)
        input_mask = ([1] * l) + ([0] * (max_d - l))
    else:
        i = i1[:max_d]
        input_mask = [1] * max_d
    
    return i, np.asarray(input_mask)

class BIN_Data_Encoder_Predict(data.Dataset):
    """Data encoder for prediction."""
    def __init__(self, list_IDs, labels, df_dti):
        self.labels = labels
        self.list_IDs = list_IDs
        self.df = df_dti
        
    def __len__(self):
        return len(self.list_IDs)

    def __getitem__(self, index):
        index = self.list_IDs[index]
        d = self.df.iloc[index]['Ligand']
        p = self.df.iloc[index]['Protein']
        
        d_v, input_mask_d = drug2emb_encoder(d)
        p_v, input_mask_p = protein2emb_encoder(p)
        y = self.labels[index]
        
        return d_v, p_v, input_mask_d, input_mask_p, y

# ==================== BEDROC calculation ====================

def calculate_bedroc(y_true, y_pred, alpha=20.0):
    """
    Calculate BEDROC (Boltzmann-Enhanced Discrimination of ROC).

    Args:
        y_true: Binary labels (0/1).
        y_pred: Prediction scores.
        alpha: Early-recognition parameter; defaults to 20.0.

    Returns:
        The BEDROC score.
    """
    # Sort by descending prediction score
    sorted_indices = np.argsort(y_pred)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    n = len(y_true)
    n_actives = np.sum(y_true)
    
    if n_actives == 0 or n_actives == n:
        return np.nan
    
    # Calculate RIE (Robust Initial Enhancement)
    ra = n_actives / n
    
    # Calculate the sum of exponentially decaying weights for active compounds
    sum_weighted_actives = 0.0
    for i, label in enumerate(sorted_labels):
        if label == 1:
            sum_weighted_actives += np.exp(-alpha * (i + 1) / n)
    
    # Calculate the random and ideal cases
    sum_random = ra * (1 - np.exp(-alpha)) / (np.exp(alpha / n) - 1)
    sum_ideal = (1 - np.exp(-alpha * ra)) / (1 - np.exp(-alpha / n))
    
    # Calculate BEDROC
    if sum_ideal - sum_random == 0:
        return np.nan
    
    bedroc = (sum_weighted_actives - sum_random) / (sum_ideal - sum_random)
    
    return bedroc

# ==================== Enrichment Factor calculation ====================

def calculate_enrichment_factor(y_true, y_pred, percentages=[0.5, 1, 2, 3, 4, 5]):
    """Calculate the Enrichment Factor (EF)."""
    sorted_indices = np.argsort(y_pred)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    total_compounds = len(y_true)
    total_actives = np.sum(y_true)
    
    if total_actives == 0:
        return {f"EF_{p}%": np.nan for p in percentages}
    
    ef_results = {}
    
    for percentage in percentages:
        n_top = max(1, int(np.ceil(total_compounds * percentage / 100.0)))
        actives_in_top = np.sum(sorted_labels[:n_top])
        expected_actives = n_top * (total_actives / total_compounds)
        
        if expected_actives > 0:
            ef = actives_in_top / expected_actives
        else:
            ef = 0.0
        
        ef_results[f"EF_{percentage}%"] = ef
        
    return ef_results

def calculate_hit_rate(y_true, y_pred, percentages=[0.5, 1, 2, 3, 4, 5]):
    """Calculate the hit rate."""
    sorted_indices = np.argsort(y_pred)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    total_compounds = len(y_true)
    total_actives = np.sum(y_true)
    
    if total_actives == 0:
        return {f"HR_{p}%": np.nan for p in percentages}
    
    hr_results = {}
    
    for percentage in percentages:
        n_top = max(1, int(np.ceil(total_compounds * percentage / 100.0)))
        actives_in_top = np.sum(sorted_labels[:n_top])
        hit_rate = actives_in_top / total_actives
        hr_results[f"HR_{percentage}%"] = hit_rate
        
    return hr_results

def calculate_precision_at_k(y_true, y_pred, percentages=[0.5, 1, 2, 3, 4, 5]):
    """Calculate Precision@k."""
    sorted_indices = np.argsort(y_pred)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    total_compounds = len(y_true)
    precision_results = {}
    
    for percentage in percentages:
        n_top = max(1, int(np.ceil(total_compounds * percentage / 100.0)))
        actives_in_top = np.sum(sorted_labels[:n_top])
        precision = actives_in_top / n_top
        precision_results[f"Precision@{percentage}%"] = precision
        
    return precision_results

# ==================== Model loading and prediction ====================

def load_best_model(model_path, device='cuda:0'):
    """Load the best_model.pt file."""
    print(f"📦 Loading model: {os.path.basename(model_path)}")
    
    config = BIN_config_DBPE()
    model = BIN_Interaction_Flat(**config)
    
    try:
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise
    
    model = model.to(device)
    model.eval()
    
    return model, config

def predict_with_padding(model, data_generator, config_batch_size, device='cuda:0', verbose=False):
    """Predict using padding."""
    y_pred = []
    y_label = []
    
    model.eval()
    total_samples = 0
    
    with torch.no_grad():
        for i, (d, p, d_mask, p_mask, label) in enumerate(data_generator):
            current_batch_size = d.size(0)
            
            if current_batch_size < config_batch_size:
                pad_size = config_batch_size - current_batch_size
                d = torch.cat([d, d[:pad_size]], dim=0)
                p = torch.cat([p, p[:pad_size]], dim=0)
                d_mask = torch.cat([d_mask, d_mask[:pad_size]], dim=0)
                p_mask = torch.cat([p_mask, p_mask[:pad_size]], dim=0)
            
            try:
                score = model(
                    d.long().to(device),
                    p.long().to(device),
                    d_mask.long().to(device),
                    p_mask.long().to(device)
                )
                
                m = torch.nn.Sigmoid()
                logits = torch.squeeze(m(score))
                
                if logits.dim() == 0:
                    logits = logits.unsqueeze(0)
                
                logits = logits[:current_batch_size].detach().cpu().numpy()
                label_ids = label.numpy()
                
                y_pred.extend(logits.flatten().tolist())
                y_label.extend(label_ids.flatten().tolist())
                
                total_samples += current_batch_size
                
            except Exception as e:
                print(f"⚠️  Error at batch {i}: {e}")
                continue
    
    if verbose:
        print(f"   Processed {total_samples} samples")
    
    return np.array(y_pred), np.array(y_label)

def calculate_metrics(y_true, y_pred, ef_percentages=[0.5, 1, 2]):
    """Calculate all evaluation metrics."""
    # Basic metrics
    auroc = roc_auc_score(y_true, y_pred)
    auprc = average_precision_score(y_true, y_pred)
    
    # BEDROC
    bedroc = calculate_bedroc(y_true, y_pred, alpha=20.0)
    
    # EF
    ef_results = calculate_enrichment_factor(y_true, y_pred, ef_percentages)
    
    metrics = {
        'ROC_AUC': auroc,
        'PR_AUC': auprc,
        'BEDROC': bedroc,
    }
    
    metrics.update(ef_results)
    
    return metrics

def evaluate_single_model(
    model_path,
    test_data_path,
    batch_size=16,
    device='cuda:0',
    kiba_threshold=3.0,
    ef_percentages=[0.5, 1, 2]
):
    """Evaluate one model."""
    
    # Load the model
    model, config = load_best_model(model_path, device)
    
    # Load the test data
    df_test = pd.read_csv(test_data_path)
    
    # Standardize column names
    if 'SMILES' in df_test.columns:
        df_test.rename(columns={'SMILES': 'Ligand'}, inplace=True)
    elif 'compound_iso_smiles' in df_test.columns:
        df_test.rename(columns={'compound_iso_smiles': 'Ligand'}, inplace=True)
    elif 'smiles' in df_test.columns:
        df_test.rename(columns={'smiles': 'Ligand'}, inplace=True)
    
    if 'target_sequence' in df_test.columns:
        df_test.rename(columns={'target_sequence': 'Protein'}, inplace=True)
    elif 'protein' in df_test.columns:
        df_test.rename(columns={'protein': 'Protein'}, inplace=True)
    
    if 'kiba' in df_test.columns and 'KIBA' not in df_test.columns:
        df_test.rename(columns={'kiba': 'KIBA'}, inplace=True)
    
    # Generate classification_label
    if 'KIBA' in df_test.columns:
        df_test['classification_label'] = (df_test['KIBA'] <= kiba_threshold).astype(int)
    else:
        raise ValueError("Cannot find KIBA column")
    
    # Create the data loader
    dataset = BIN_Data_Encoder_Predict(
        df_test.index.values,
        df_test.classification_label.values,
        df_test
    )
    
    data_loader = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False
    )
    
    # Predict
    predictions, labels = predict_with_padding(
        model, data_loader, config['batch_size'], device, verbose=False
    )
    
    # Calculate metrics
    metrics = calculate_metrics(labels, predictions, ef_percentages)
    
    return metrics

def evaluate_multiple_models(
    model_dir,
    model_pattern,
    model_ids,
    test_data_path,
    output_summary_path=None,
    batch_size=16,
    device='cuda:0',
    kiba_threshold=3.0,
    ef_percentages=[0.5, 1, 2]
):
    """
    Evaluate multiple models and calculate means and standard deviations.

    Args:
        model_dir: Directory containing the models.
        model_pattern: Filename template, such as '10_50_{}_best_model.pt'.
        model_ids: Model IDs, such as [1, 2, 3, 4, 5].
        test_data_path: Path to the test data.
        output_summary_path: Path for the result table.
        ef_percentages: Percentages at which EF is calculated.
    """
    
    print("="*80)
    print("🚀 Evaluating Multiple Models")
    print("="*80)
    
    # Check the device
    if not torch.cuda.is_available() and device.startswith('cuda'):
        print("⚠️  CUDA not available, using CPU")
        device = 'cpu'
    print(f"💻 Using device: {device}")
    print(f"📊 KIBA threshold: ≤ {kiba_threshold}")
    print(f"📈 EF percentages: {ef_percentages}")
    print(f"🔢 Number of models: {len(model_ids)}")
    print()
    
    # Store results for all models
    all_results = []
    
    # Iterate over models
    for model_id in model_ids:
        model_filename = model_pattern.format(model_id)
        model_path = os.path.join(model_dir, model_filename)
        
        if not os.path.exists(model_path):
            print(f"⚠️  Model not found: {model_path}")
            continue
        
        print(f"📦 Evaluating Model {model_id}/{len(model_ids)}: {model_filename}")
        
        try:
            metrics = evaluate_single_model(
                model_path=model_path,
                test_data_path=test_data_path,
                batch_size=batch_size,
                device=device,
                kiba_threshold=kiba_threshold,
                ef_percentages=ef_percentages
            )
            
            # Add the model ID
            metrics['model_id'] = model_id
            all_results.append(metrics)
            
            # Print results for the current model
            print(f"   ✅ EF_0.5%: {metrics['EF_0.5%']:.2f} | "
                  f"EF_1%: {metrics['EF_1%']:.2f} | "
                  f"EF_2%: {metrics['EF_2%']:.2f} | "
                  f"ROC_AUC: {metrics['ROC_AUC']:.4f} | "
                  f"PR_AUC: {metrics['PR_AUC']:.4f} | "
                  f"BEDROC: {metrics['BEDROC']:.4f}")
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
        
        print()
    
    if len(all_results) == 0:
        print("❌ No models were successfully evaluated!")
        return None
    
    # Convert to a DataFrame
    df_results = pd.DataFrame(all_results)
    
    # Calculate aggregate statistics
    print("="*80)
    print("📊 Summary Statistics")
    print("="*80)
    print()
    
    metric_cols = ['EF_0.5%', 'EF_1%', 'EF_2%', 'ROC_AUC', 'PR_AUC', 'BEDROC']
    
    # Create the result table
    summary_data = []
    
    print(f"{'Metric':<15} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-"*80)
    
    for metric in metric_cols:
        if metric in df_results.columns:
            mean_val = df_results[metric].mean()
            std_val = df_results[metric].std()
            min_val = df_results[metric].min()
            max_val = df_results[metric].max()
            
            print(f"{metric:<15} {mean_val:<12.4f} {std_val:<12.4f} {min_val:<12.4f} {max_val:<12.4f}")
            
            summary_data.append({
                'Metric': metric,
                'Mean': mean_val,
                'Std': std_val,
                'Min': min_val,
                'Max': max_val
            })
    
    print()
    
    # Detailed result table
    print("="*80)
    print("📋 Detailed Results for Each Model")
    print("="*80)
    print()
    
    print(df_results[['model_id'] + metric_cols].to_string(index=False))
    print()
    
    # Save results
    if output_summary_path is None:
        output_summary_path = os.path.join(model_dir, 'evaluation_summary.csv')
    
    # Save detailed results
    detailed_path = output_summary_path.replace('.csv', '_detailed.csv')
    df_results.to_csv(detailed_path, index=False)
    print(f"💾 Detailed results saved to: {detailed_path}")
    
    # Save aggregate statistics
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv(output_summary_path, index=False)
    print(f"💾 Summary statistics saved to: {output_summary_path}")
    
    print("\n" + "="*80)
    print("✅ Evaluation Completed Successfully!")
    print("="*80)
    
    return df_results, df_summary

# ==================== Usage example ====================

if __name__ == '__main__':
    
    # Configuration
    model_dir = _kp('model/MolTrans')
    model_pattern = '10_50_{}_best_model.pt'  # {} is replaced with 1, 2, 3, 4, or 5
    model_ids = [1, 2, 3, 4, 5]  # Model IDs
    
    test_data_path = _kp('dataset/KIBA_seen_LUDe_augment_sim08.csv')
    
    output_summary_path = _kp('others/baseline/MolTrans/results/KIBA_seen_LUDe_augment/KIBA_seen_LUDe_augment_sim08_result_overall.csv')
    
    # Run the evaluation
    df_results, df_summary = evaluate_multiple_models(
        model_dir=model_dir,
        model_pattern=model_pattern,
        model_ids=model_ids,
        test_data_path=test_data_path,
        output_summary_path=output_summary_path,
        batch_size=16,
        device='cuda:0',
        kiba_threshold=3.0,  # KIBA <= 3 indicates an active compound
        ef_percentages=[0.5, 1, 2]  # Calculate EF_0.5%, EF_1%, and EF_2%
    )
